In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Set, Dict
from collections import defaultdict
import os

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [22]:
bace_df = pd.read_csv("../bace.csv").sample(frac=1)

In [23]:
bace_dictionary = {}

for i, row in bace_df.iterrows():
    mol = Chem.MolFromSmiles(row["mol"])

    lfg = rdMolStandardize.LargestFragmentChooser()
    mol = lfg.choose(mol)

    uncharger = rdMolStandardize.Uncharger()
    mol = uncharger.uncharge(mol)

    smiles = Chem.MolToSmiles(mol)

    smiles_value = bace_dictionary.get(smiles)
    if smiles_value == None:
        bace_dictionary[smiles] = row["pIC50"]

In [24]:
scaffold_dictonary = defaultdict(list)

for smiles, pIC50 in bace_dictionary.items():
    mol = Chem.MolFromSmiles(smiles)

    try: # From what I've understood, there are atoms with more than 4 bonds in the dataset, catch these errors and skip them.
        scaffold = MurckoScaffold.MakeScaffoldGeneric(mol)
    except Chem.AtomValenceException:
        continue

    scaffold_smiles = Chem.MolToSmiles(scaffold, canonical=True)

    if scaffold_smiles != '' and scaffold.GetNumAtoms() > 0:
        scaffold_dictonary[scaffold_smiles].append((smiles, pIC50))

In [25]:
heldout_molecule_target = 200

heldout_data = []
remaining_molecules_dictionary = defaultdict(list)

for scaffold, molecules in scaffold_dictonary.items():
    if len(heldout_data) + len(molecules) <= heldout_molecule_target:
        heldout_data.extend(molecules)
    else:
        remaining_molecules_dictionary[scaffold] = molecules

directory = f"heldout_datasets"
if not os.path.exists(directory): os.makedirs(directory)
fold_df = pd.DataFrame(data=heldout_data, columns=["smiles", "pIC50"])
fold_df.to_csv(f"{directory}/heldout_testset.csv")

In [26]:
pure_datasets_molecule_targets = [1300, 900, 500]

pure_datasets_dictionaries = []

for target in pure_datasets_molecule_targets:
    dataset = defaultdict(list)
    molecule_count = 0
    for scaffold, molecules in remaining_molecules_dictionary.items():
        if molecule_count + len(molecules) <= target:
            dataset[scaffold] = molecules
            molecule_count += len(molecules)
    pure_datasets_dictionaries.append(dataset)

In [ ]:
#scaffold_datasets = []
#
#for combination in processed_data_combinations:
#    dataset = defaultdict(list)
#    for smiles, logP in combination.items():
#        mol = Chem.MolFromSmiles(smiles)
#
#        try: # From what I've understood, there are atoms with more than 4 bonds in the dataset, catch these errors and skip them.
#            scaffold = MurckoScaffold.MakeScaffoldGeneric(mol)
#        except Chem.AtomValenceException:
#            continue
#
#        scaffold_smiles = Chem.MolToSmiles(scaffold, canonical=True)
#
#        if scaffold_smiles != '' and scaffold.GetNumAtoms() > 0:
#                dataset[scaffold_smiles].append(smiles)
#    scaffold_datasets.append(dataset)

In [27]:
for dataset_index in range(len(pure_datasets_molecule_targets)):
    folds_data = [[], [], [], [], []]
    fold_size = pure_datasets_molecule_targets[dataset_index] / len(folds_data)

    for scaffold, molecules in pure_datasets_dictionaries[dataset_index].items():
        for fold_index in range(len(folds_data)):
            if len(folds_data[fold_index]) + len(molecules) <= fold_size:
                folds_data[fold_index].extend(molecules)
                break

    directory = f"combination_{pure_datasets_molecule_targets[dataset_index]}_molecules_and_0_%_synthetic"
    for fold_index in range(len(folds_data)):
        if not os.path.exists(directory): os.makedirs(directory)
        fold_df = pd.DataFrame(data=folds_data[fold_index], columns=["smiles", "pIC50"])
        fold_df.to_csv(f"{directory}/fold_{fold_index}.csv")